# Create Carnegie Corporation of New York Awards (GRANT PATTERN, paginated listing scrape)

Carnegie Corporation of New York (est. 1911) funds organizations directly — universities, think tanks, NGOs — across four program areas: Education, Democracy, International Program, and Special Projects.

**Awarding body:** Carnegie Corporation of New York - F4320306125 (US, DOI 10.13039/100000308)

**Source:** Paginated grants database at carnegie.org/grants/grants-database/?page=N (pages 1..372, ~25 grants/page). Each listing card carries the full per-grant record inline — Year, Grantee Org, $Amount, Program. No per-grant detail fetch needed.

**Important caveat — re-test Cloudflare WAF before deploy:** CLAUDE.md's burned-source list documents Carnegie as Cloudflare-403-blocked. On 2026-05-27 all my probes returned HTTP 200 cleanly with the full listing markup, so the WAF either relaxed the rule or never applied to my User-Agent. If the admin's Databricks run hits 403 during re-ingest, fall back to Method 6 (Playwright headless Chrome).

**Schema choices:**
- One row per grant. `funder_award_id` = `carnegie-{grant_id}` where grant_id is the trailing fragment of the source detail URL (`/grants-database/grant/{ID}/`).
- `funder_scheme` = source `program` field (Education / Democracy / International Program / Special Projects / Legacy / + a few smaller programs).
- `funding_type` = `grant` for all rows. Carnegie's program structure is org-level grantmaking (not research / fellowship / prize).
- `amount` populated from the listing card's `$X,YYY` field (100% coverage in smoke + full run); `currency` = `USD`.
- `start_year` from the listing card's Year field.
- `lead_investigator` shaped as ORG-only — Carnegie funds organizations, not PIs: given/family/orcid NULL, `affiliation.name` = grantee org. `affiliation.country` = NULL because the source does not expose grantee country per grant (Carnegie funds US + international grantees, and we don't infer from name).
- `description` ships NULL — the source listing doesn't include a description; only the detail-page narrative does, and per-grant detail fetches across 9K+ records was deemed not worth the added scrape time for this PR (a follow-up can backfill).

**S3 location:** `s3a://openalex-ingest/awards/carnegie/carnegie_grants.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.carnegie_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/carnegie/carnegie_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.carnegie_raw;

## Step 1.5: Inspect raw + per-program/year distribution

Expected: ~9,280 rows (372 listing pages × ~25 cards/page, last page truncated to 5). 100% coverage on grantee_org + year + amount + program. Year range likely spans ~1995-2026.

In [ ]:
%sql
DESCRIBE openalex.awards.carnegie_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.carnegie_raw LIMIT 5;

In [ ]:
%sql
-- Program distribution.
SELECT program, COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 0) AS avg_amount
FROM openalex.awards.carnegie_raw
GROUP BY program
ORDER BY rows DESC
LIMIT 15;

In [ ]:
%sql
-- Year range.
SELECT MIN(year) AS min_year, MAX(year) AS max_year, COUNT(DISTINCT year) AS years
FROM openalex.awards.carnegie_raw
WHERE year IS NOT NULL;

## Step 1.6: Fail-fast - verify Carnegie funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306125;

## Step 2: Transform to award schema

`display_name` = `Carnegie {program} - {grantee_org} ({year})`. `description` ships NULL (source listing has no per-grant description; detail-page narrative not scraped). `lead_investigator` shaped as org-only (given/family/orcid NULL, affiliation.name = grantee_org, affiliation.country = NULL since the source does not expose grantee country per grant).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.carnegie_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306125  -- Carnegie Corporation of New York
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('Carnegie ', COALESCE(n.program, 'Grant'), ' - ', n.grantee_org,
           CASE WHEN n.year IS NOT NULL THEN CONCAT(' (', n.year, ')') ELSE '' END) AS display_name,
    CAST(NULL AS STRING) AS description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    n.program AS funder_scheme,
    'carnegie_corporation' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.grantee_org IS NULL OR n.grantee_org = '' THEN NULL
        ELSE struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.grantee_org AS name,
                CAST(NULL AS STRING) AS country,  -- source doesn't expose grantee country per grant
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.carnegie_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.grantee_org IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 141

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'carnegie_corporation' AND priority = 141;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    141 AS priority  -- Carnegie Corporation of New York
FROM openalex.awards.carnegie_awards;

## Step 6: Verification

Expect ~9,280 rows, 100% amount (USD-only), 100% grantee_org + start_year + funder_scheme. funding_type=grant for all rows. description NULL across the board (documented above).

In [ ]:
%sql
SELECT COUNT(*) AS total_carnegie_rows FROM openalex.awards.carnegie_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.carnegie_awards;

In [ ]:
%sql
-- §6.3 Data completeness. pct_description expected 0 (source listing has no description field).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_grantee,
    COUNT(start_year) AS has_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.carnegie_awards;

In [ ]:
%sql
-- §6.7 amount check. Expect 100% USD; range and median useful for sanity.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.carnegie_awards;

In [ ]:
%sql
-- Year distribution (totals per year).
SELECT start_year, COUNT(*) AS rows, ROUND(SUM(amount), 0) AS yearly_total_usd
FROM openalex.awards.carnegie_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.carnegie_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows.
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       funder_scheme AS program, start_year, amount,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS grantee_org
FROM openalex.awards.carnegie_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;